# Quick Start

ApertureDB stores image bytes together with metadata and retrieves them through JSON queries. This example stores one ramen photograph, finds it with a metadata filter, and removes the record it created. It requires no embedding model or external AI credential.

The example requires Python 3.10 or later, internet access for the SDK and image, and an ApertureDB instance with permission to add, read, and delete images. For an existing [Cloud instance](https://docs.aperturedata.io/Setup/server/Cloud), configure `APERTUREDB_KEY` in your environment before launching Jupyter. A saved [connection profile](https://docs.aperturedata.io/Setup/client/configuration) also works. Keep credentials out of notebook cells and saved outputs.

For local evaluation, complete the [deployment setup](https://docs.aperturedata.io/administration/deployment/DockerCompose) and configure its connection profile. Plaintext evaluation connections must remain on a trusted local machine. This notebook connects to a running server.



In [ ]:
%pip install --quiet "aperturedb>=1.0,<2"


<a id="connect-and-verify"></a>

## Connect to ApertureDB

`create_connector()` reads your environment or active profile. `GetStatus` confirms that the client can reach and authenticate to the selected database before sending a write.


In [ ]:
from aperturedb.CommonLibrary import create_connector
from importlib.metadata import version

client = create_connector()
status, _ = client.query([{"GetStatus": {}}])
assert isinstance(status, list) and status[0]["GetStatus"]["status"] == 0, status
print("Connected; SDK", version("aperturedb"))


<a id="load-the-cookbook-dataset"></a>

## Load one image

`AddImage` stores the ramen photograph with a `dish` property and a unique `tutorial_run` identifier. The identifier separates this run's record from other data. The photograph comes from a fixed Cookbook revision, and its SHA-256 digest identifies the expected download.


In [ ]:
from hashlib import sha256
from urllib.request import urlopen
from uuid import uuid4

fixture_url = (
    "https://raw.githubusercontent.com/aperture-data/Cookbook/"
    "51b3c3d9f495f8fd77108a270b88fe8ab8cc0004/images/ramen.jpg"
)
with urlopen(fixture_url, timeout=30) as download:
    image_bytes = download.read()
assert sha256(image_bytes).hexdigest() == "886652b23889c16608ec0557536f6f0bd32d5d3f0d750f5c185044f9ffed3dd1", "Unexpected image fixture"
run_id = "quickstart-" + uuid4().hex
query = [{"AddImage": {"class": 'Images', 
    "properties": {"tutorial_run": run_id, "dish": "ramen"}
}}]
response, _ = client.query(query, [image_bytes])
assert isinstance(response, list) and response[0]["AddImage"]["status"] == 0, response
print("Added one image")


<a id="a-find-images-of-dishes-from-scottish-cuisine"></a>
<a id="run-queries"></a>

## Run a filtered query

The expression array compares `tutorial_run` with this run's identifier. `results.list` requests metadata, and `blobs: True` includes the image bytes. The response contains one image with `dish: ramen`, which the cell displays.


In [ ]:
from IPython.display import Image, display

query = [{"FindImage": {
    "constraints": ["$tutorial_run", "==", run_id],
    "blobs": True,
    "results": {"list": ["dish", "tutorial_run"]}
}}]
response, blobs = client.query(query)
assert isinstance(response, list), response
result = response[0]["FindImage"]
assert result["status"] == 0 and result["returned"] == 1, response
assert result["entities"][0]["dish"] == "ramen", result
assert result["entities"][0]["tutorial_run"] == run_id, result
assert len(blobs) == 1 and blobs[0], "Expected one image buffer"
display(Image(data=blobs[0]))
print("Retrieved the expected ramen image")


## Clean up and rerun

The cleanup query deletes the image selected by `tutorial_run`; the following count query returns zero matches. Run cleanup before restarting the notebook. If a run is interrupted, its `tutorial_run` value identifies the leftover record without selecting unrelated images.


In [ ]:
response, _ = client.query([{"DeleteImage": {
    "constraints": ["$tutorial_run", "==", run_id]
}}])
assert isinstance(response, list) and response[0]["DeleteImage"]["status"] == 0, response
response, _ = client.query([{"FindImage": {
    "constraints": ["$tutorial_run", "==", run_id],
    "results": {"count": True}
}}])
assert response[0]["FindImage"]["status"] == 0, response
assert response[0]["FindImage"]["count"] == 0, response
print("Cleanup verified")


## Troubleshooting

- Connection or authentication failure: check the active profile, endpoint, port,
  credentials, and TLS setting in [client setup](https://docs.aperturedata.io/Setup/client/adb).
- Permission failure: ask the database administrator for image read, add, and
  delete permissions; Cloud account membership and database permissions differ.
- Download checksum failure: stop before ingestion and check the download. Do not
  bypass the checksum to accept an HTML error page as an image.
- No image buffer: verify `blobs: True` and inspect the command status before
  indexing the returned buffers.

<a id="b-find-images-and-information-of-dishes-with-butter-and-chicken-in-them"></a>
<a id="c-find-similar-looking-dishes-to-a-bread"></a>

## What's Next?

Related examples cover [images](https://docs.aperturedata.io/HowToGuides/start/Images),
[graph connections](https://docs.aperturedata.io/HowToGuides/start/Connections),
and [vector search](https://docs.aperturedata.io/HowToGuides/start/VectorSearch).
The [query language overview](https://docs.aperturedata.io/query_language/Overview/Introduction)
explains how filters, references, and result projection fit together.
